# 원판 안쪽, 바로 바깥, 경계를 가리면 어떻게 달라질까?

같은 검증 프레임 a03 19번과 b02 30번을 다시 봅니다. 두 프레임의 오른쪽 하단에서 허브와 다리를 피할 수 있는 구간을 먼저 골랐습니다. 같은 바닥 질감 16×10픽셀을 두 프레임의 세 위치에 그대로 붙입니다. 원래 정답 박스에 대한 패치 크기 비율은 서로 다릅니다. 지난 20% 실험에서 쓴 넓은 패치도 프레임별로 보존해 별도로 비교합니다. 넓은 패치의 안쪽 조건은 허브 일부도 덮습니다.

주황 점선은 **사람이 확대해 읽은 원본의 윤곽**입니다. 가로 위치마다 윤곽의 높이가 달라 직사각형 박스의 아래 변만으로 안팎을 정할 수 없습니다. 세로로 조금씩 이동해 안쪽 조건은 전부 원판 쪽, 바깥 조건은 전부 배경 쪽, 경계 조건은 대략 절반씩 놓습니다. 흐림과 다리·그림자가 겹치는 구간의 선은 근사치입니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ultralytics import YOLO

from wlcv.config import DATASET, RUNS
from wlcv.evaluation.patch_position import reviewed_floor_frames
from wlcv.evaluation.rim_boundary import SIZES, prepare_cases, measure

report = RUNS / "plate_baseline/report"
out = report / "rim_boundary"
out.mkdir(parents=True, exist_ok=True)
floors = reviewed_floor_frames(report / "occlusion_half_area/mask_positions.csv")
previous = pd.read_csv(report / "occlusion_dose/per_frame.csv")
cases = prepare_cases(DATASET, floors, previous)

## 1. 원본과 실제로 가리는 픽셀부터 확인

아래 그림의 열 순서는 원본·안쪽·바깥·경계입니다. 작은 패치 그림은 각 프레임 아래에 동일한 위치의 확대본을 붙입니다. 청록색 사각형과 주황색 선은 화면에만 표시합니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from wlcv.evaluation.patch_position import paste_patch
from wlcv.evaluation.rim_boundary import CONDITIONS, PROFILES


def compare_figure(cases, size, scores=None):
    small = size == "Small"
    fig = plt.figure(figsize=(11, 8.4 if small else 6.9), layout="constrained")
    grid = fig.add_gridspec(2, 4)
    for row, stem in enumerate(PROFILES):
        case = cases[stem, size]
        x1, y1, x2, y2 = case["truth"]
        width, height = x2 - x1, y2 - y1
        outside_x, outside_y, patch = case["placements"]["Outside"]
        h, w = patch.shape[:2]
        edge_x, edge_y = np.array(PROFILES[stem]).T
        for col, condition in enumerate(CONDITIONS):
            sub = grid[row, col].subgridspec(2 if small else 1, 1, height_ratios=[2.8, 1.25] if small else [1])
            full = fig.add_subplot(sub[0])
            axes = [full, fig.add_subplot(sub[1])] if small else [full]
            placement = case["placements"][condition]
            input_image = paste_patch(case["image"], placement)
            for ax in axes:
                ax.imshow(input_image, interpolation="nearest")
                if placement is None:
                    ax.plot(edge_x, edge_y, color="#ff844f", ls="--", lw=1)
                else:
                    x, y, _ = placement
                    ax.add_patch(Rectangle((x-.5, y-.5), w, h, fill=False, edgecolor="#15d5df",
                                           linewidth=.85))
                ax.set(xticks=[], yticks=[])
                for spine in ax.spines.values():
                    spine.set_visible(False)
            full.set(xlim=(x1-.12*width, x2+.12*width),
                     ylim=(max(y2+.10*height, outside_y+h+.05*height), y1-.08*height))
            if small:
                zx = outside_x + w/2
                zy = np.interp(zx, edge_x, edge_y)
                axes[-1].set(xlim=(zx-40, zx+40), ylim=(zy+21, zy-21))
            if row == 0:
                full.set_title({"Original":"Original", "Inside":"Inside rim", "Outside":"Outside rim",
                                "Boundary":"Across boundary"}[condition], fontsize=14)
            if col == 0:
                person, frame = stem.split("_")[0], stem.rsplit("f", 1)[-1]
                full.set_ylabel(f"{person}  /  frame {frame}", fontsize=12)
            if scores is not None:
                hit = scores.loc[stem, size, condition]
                label = (f"Conf {hit.confidence:.3f}   IoU {hit.iou:.3f}"
                         if np.isfinite(hit.confidence) else "No IoU match")
                axes[-1].set_xlabel(label, fontsize=10, color="#1e504b" if hit.detected else "#9c361d")
    title = "One shared 16 × 10 px patch" if small else "Original 20% patches"
    fig.suptitle(title + "   |   Plate, adjacent background and boundary", fontsize=16)
    return fig

In [ ]:
for size in SIZES:
    compare_figure(cases, size)
    plt.show()

## 2. 저장된 베이스라인으로 추론

전체 해상도 프레임을 모델에 전달합니다. Mac CPU, 입력 크기 640, 후보 신뢰도 하한 0.001입니다. 원래 정답과 IoU가 0.50 이상인 원판 후보 중 최고 신뢰도와 그 후보의 IoU를 기록합니다. 해당 후보가 없으면 두 값은 결측입니다. 신뢰도 0.10 이상을 검출로 셉니다. 실제 원본 추론은 프레임당 한 번입니다.

In [ ]:
model = YOLO(str((RUNS / "plate_baseline/weights/best.pt").resolve(strict=True)))
assert model.names == {0: "plate"}
results = measure(model, cases)
originals = results.query("size == 'Small' and condition == 'Original'").set_index("stem")
saved = previous.query("model == 'Baseline' and dose == 0 and position == 'Center'").set_index("stem")
np.testing.assert_allclose(originals[["confidence", "iou"]], saved.loc[originals.index, ["confidence", "iou"]],
                           rtol=0, atol=1e-6)
results.to_csv(out / "per_frame.csv", index=False)
display(results[["stem", "size", "condition", "width", "height", "pct_box_area", "pct_patch_inside",
                 "confidence", "iou", "detected"]].round(3))

## 3. 후보의 점수와 위치를 함께 읽기

같은 행의 원본과 가림 조건을 비교하세요. 작은 패치는 두 프레임에서도 같은 픽셀이며, 넓은 패치의 질감은 각 프레임 안에서 같습니다. 표의 `pct_patch_inside`는 수동 윤곽으로 추정한 **패치 면적 중 원판 쪽에 놓인 비율**입니다. `pct_box_area`는 **정답 박스에 대한 전체 패치의 면적 비율**입니다.

In [ ]:
scores = results.set_index(["stem", "size", "condition"])
for size, filename in [("Small", "shared_patch.png"), ("Previous20", "previous_20pct.png")]:
    fig = compare_figure(cases, size, scores)
    fig.savefig(out / filename, dpi=150, facecolor="white")
    plt.show()